# Análise Exploratória de Dados - BORALI

**História 3.1 - Análise Exploratória**  
Dataset: Pesquisa de campo com 45 respondentes sobre descoberta de eventos culturais no Recife (mar/2026)

## Objetivos
- Investigar distribuição do perfil dos usuários
- Identificar padrões de comportamento e preferências
- Gerar gráficos descritivos para comunicar insights
- Documentar achados para o sistema de recomendação


In [ ]:
!git clone https://github.com/Sofia1653/BORALI.git
import os
os.chdir('BORALI/data/notebooks')

---
## 0. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

PROCESSED = Path('../processed')
FIGURES   = Path('../figures')
FIGURES.mkdir(exist_ok=True)

BLUE   = '#3266ad'
TEAL   = '#2d9674'
CORAL  = '#b85c2a'
PURPLE = '#8c5db5'
GRAY   = '#73726c'
PALETTE = [BLUE, TEAL, CORAL, PURPLE, GRAY]

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

df = pd.read_csv(PROCESSED / 'eventos_culturais_recife_tratado.csv')
print(f'Shape: {df.shape}')
df.head(3)

---
## 1. Visão geral do dataset

In [ ]:
print('=== Tipos de dados e valores nulos ===')
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'nulos': df.isnull().sum(),
    '% nulos': (df.isnull().sum() / len(df) * 100).round(1)
})
print(summary)

In [ ]:
df[['num_plataformas']].describe()

---
## 2. Perfil dos respondentes

In [ ]:
#Gráfico 1: Faixa etária
faixa_order = ['16-24', '25-39', '40-59', '60+']
faixa_counts = df['faixa_etaria'].value_counts().reindex(faixa_order)

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    faixa_counts,
    labels=faixa_counts.index,
    autopct='%1.0f%%',
    colors=PALETTE,
    startangle=90,
    wedgeprops={'width': 0.55},
)
for t in autotexts:
    t.set_fontsize(11)
ax.set_title('Faixa etária dos respondentes', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(FIGURES / 'g1_faixa_etaria.png', bbox_inches='tight')
plt.show()

print(faixa_counts.to_string())

In [ ]:
#Gráfico 2: Frequência de participação
freq_order = ['Nunca', 'Raramente', 'Às vezes', 'Frequentemente', 'Muito frequentemente']
freq_counts = df['frequencia_eventos'].value_counts().reindex(freq_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(freq_counts.index, freq_counts.values, color=BLUE, width=0.55, zorder=2)
ax.bar_label(bars, padding=3, fontsize=11)
ax.set_title('Com que frequência você participa de eventos culturais?', fontsize=12)
ax.set_ylabel('Respondentes')
ax.set_ylim(0, freq_counts.max() + 5)
plt.tight_layout()
plt.savefig(FIGURES / 'g2_frequencia_eventos.png', bbox_inches='tight')
plt.show()

---
## 3. Preferências e comportamentos

In [ ]:
#Gráfico 3: Tipos de eventos (múltipla escolha)
tipos = (
    df['tipos_eventos']
    .dropna()
    .str.split('; ')
    .explode()
    .str.strip()
    .value_counts()
    .drop('Não costumo participar', errors='ignore')
)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(tipos.index[::-1], tipos.values[::-1], color=TEAL, zorder=2)
ax.bar_label(bars, padding=4, fontsize=11)
ax.set_title('Quais tipos de eventos você costuma frequentar? (múltipla escolha)', fontsize=12)
ax.set_xlabel('Menções (n=45)')
ax.set_xlim(0, tipos.max() + 6)
ax.grid(axis='y', alpha=0)
plt.tight_layout()
plt.savefig(FIGURES / 'g3_tipos_eventos.png', bbox_inches='tight')
plt.show()

print(tipos)

In [ ]:
#Gráfico 4: Canais de descoberta
canais = (
    df['canais_descoberta']
    .dropna()
    .str.split('; ')
    .explode()
    .str.strip()
    .value_counts()
)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(canais.index[::-1], canais.values[::-1], color=PURPLE, zorder=2)
ax.bar_label(bars, padding=4, fontsize=11)
ax.set_title('Como você costuma descobrir eventos culturais? (múltipla escolha)', fontsize=12)
ax.set_xlabel('Menções (n=45)')
ax.set_xlim(0, canais.max() + 8)
ax.grid(axis='y', alpha=0)
plt.tight_layout()
plt.savefig(FIGURES / 'g4_canais_descoberta.png', bbox_inches='tight')
plt.show()

print(canais)

---
## 4. Percepção sobre a plataforma e o app

In [ ]:
#Gráfico 5: Facilidade de descoberta
facil_order = ['Muito difícil', 'Difícil', 'Neutro', 'Fácil', 'Muito fácil']
facil_colors = [CORAL, '#d4855a', GRAY, TEAL, '#1d5fa5']
facil_counts = df['facilidade_descoberta'].value_counts().reindex(facil_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(facil_counts.index, facil_counts.values, color=facil_colors, width=0.55, zorder=2)
ax.bar_label(bars, padding=3, fontsize=11)
ax.set_title('Você considera fácil encontrar eventos que combinam com seu gosto?', fontsize=11)
ax.set_ylabel('Respondentes')
ax.set_ylim(0, facil_counts.max() + 4)
plt.tight_layout()
plt.savefig(FIGURES / 'g5_facilidade_descoberta.png', bbox_inches='tight')
plt.show()

print(facil_counts)

In [ ]:
#Gráfico 6: Utilidade do app de recomendação
util_order = ['Pouco útil', 'Neutro', 'Útil', 'Muito útil']
util_colors = [CORAL, GRAY, '#3b81c0', BLUE]
util_counts = df['utilidade_recomendacao_app'].value_counts().reindex(util_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(util_counts.index, util_counts.values, color=util_colors, width=0.5, zorder=2)
ax.bar_label(bars, padding=3, fontsize=11)
ax.set_title('Se o app recomendasse eventos por interesse, acharia útil?', fontsize=11)
ax.set_ylabel('Respondentes')
ax.set_ylim(0, util_counts.max() + 4)
plt.tight_layout()
plt.savefig(FIGURES / 'g6_utilidade_app.png', bbox_inches='tight')
plt.show()

In [ ]:
#Gráfico 7: Funcionalidades mais desejadas
funcs = (
    df['funcionalidades_uteis']
    .dropna()
    .str.split('; ')
    .explode()
    .str.strip()
    .value_counts()
)

# Rótulos curtos para o gráfico
label_map = {
    'Recomendações baseadas nos meus interesses': 'Recomendações por interesse',
    'Eventos perto de mim': 'Eventos próximos',
    'Mapa com eventos da cidade': 'Mapa de eventos',
    'Descoberta de eventos gratuitos': 'Gratuitos',
    'Notificações sobre eventos próximos': 'Notificações',
    'Agenda cultural personalizada': 'Agenda personalizada',
}
funcs.index = [label_map.get(i, i) for i in funcs.index]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(funcs.index[::-1], funcs.values[::-1], color=BLUE, zorder=2)
ax.bar_label(bars, padding=4, fontsize=11)
ax.set_title('Funcionalidades mais desejadas no app (múltipla escolha)', fontsize=12)
ax.set_xlabel('Menções (n=45)')
ax.set_xlim(0, funcs.max() + 6)
ax.grid(axis='y', alpha=0)
plt.tight_layout()
plt.savefig(FIGURES / 'g7_funcionalidades.png', bbox_inches='tight')
plt.show()

print(funcs)

---
## 5. Análise de dor: percepção de cobertura

In [ ]:
#Gráfico 8: Escala Likert - pouca divulgação de eventos gratuitos vs. poucos eventos no bairro
likert_order = ['Discordo totalmente', 'Discordo', 'Neutro', 'Concordo', 'Concordo totalmente']
likert_colors = ['#1d5fa5', '#3b81c0', GRAY, '#d4855a', CORAL]

d1 = df['pouca_divulgacao_gratuitos'].value_counts().reindex(likert_order).fillna(0)
d2 = df['poucos_eventos_bairro'].value_counts().reindex(likert_order).fillna(0)

x = np.arange(len(likert_order))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4.5))
b1 = ax.bar(x - width/2, d1, width, label='Pouca divulgação de gratuitos', color=TEAL, zorder=2)
b2 = ax.bar(x + width/2, d2, width, label='Poucos eventos no bairro', color=CORAL, zorder=2)
ax.bar_label(b1, padding=2, fontsize=10)
ax.bar_label(b2, padding=2, fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(likert_order, fontsize=10)
ax.set_ylabel('Respondentes')
ax.set_title('Percepção sobre cobertura e divulgação de eventos', fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, max(d1.max(), d2.max()) + 5)
plt.tight_layout()
plt.savefig(FIGURES / 'g8_cobertura_likert.png', bbox_inches='tight')
plt.show()

In [ ]:
#Gráfico 9: Bairros mais representados
bairros = df['bairro'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(bairros.index[::-1], bairros.values[::-1], color=PURPLE, zorder=2)
ax.bar_label(bars, padding=4, fontsize=11)
ax.set_title('Top 10 bairros/regiões dos respondentes', fontsize=12)
ax.set_xlabel('Respondentes')
ax.set_xlim(0, bairros.max() + 2)
ax.grid(axis='y', alpha=0)
plt.tight_layout()
plt.savefig(FIGURES / 'g9_bairros.png', bbox_inches='tight')
plt.show()

---
## 6. Análise bivariada

In [ ]:
# Frequência de eventos vs. facilidade de descoberta
pivot = pd.crosstab(
    df['frequencia_eventos'],
    df['facilidade_descoberta']
)
print('Frequência de eventos × Facilidade de descoberta')
print(pivot)

In [ ]:
# Faixa etária vs. usaria app de localização
pivot2 = pd.crosstab(
    df['faixa_etaria'],
    df['usaria_app_localizacao'],
    normalize='index'
).round(2)
print('Faixa etária × Usaria app (proporção por linha)')
print(pivot2)

In [ ]:
# Número de plataformas vs. foi a evento por acaso
print('Número de plataformas × Já foi a evento por acaso')
print(pd.crosstab(df['num_plataformas'], df['foi_evento_acaso']))

---
## 7. Resumo dos insights

### KPIs calculados a partir da amostra

| Métrica | Valor | Interpretação |
|---|---|---|
| Taxa de usuários que perderam eventos | **95%** | Dor central do produto |
| Taxa de aceitação do app ("certeza"+"provável") | **78%** | Alta receptividade |
| Taxa de utilidade do sistema de recomendação | **89%** | Valida o core do BORALI |
| Taxa de descoberta por acaso | **84%** | Oportunidade para notificações |
| % que considera descoberta difícil/muito difícil | **42%** | + 36% neutro = 78% não satisfeitos |
| Canal dominante de descoberta | **Instagram (93%)** | Integração social é crítica |
| Evento mais popular | **Cinema (82%)** | Categoria prioritária para o catálogo |
| Funcionalidade mais desejada | **Recomendações (84%)** | Reforça eixo de personalização |
